# 15 — Time-forward validation + the test-separator effect → final VFM model F2

The judges will run the app on **new test data** (most likely Oct–Dec 2025, after our data ends). So the model
must **extrapolate forward in time**, not just interpolate between tests (what leave-one-test-out, LOTO, measures).

**Time-forward test** (`validation.time_forward`): train only on tests **before** a cutoff, predict the tests in the
next 92 days. Two cutoffs (1 Jun, 1 Jul) → 56 forecasts, errors also per horizon (0–30, 30–60, 60–92 days).
Pessimistic: here the model learns from 48–57 tests (5–6 months); for Oct–Dec it learns from all 85 (9 months).

Forecasting remedies tested for the physics trends:
- **slope shrinkage** — pull each well's P_res slope toward the field median: `slope = (1−s)·own + s·median`
- **recency weighting** — weight tests by `exp((t − t_last)/τ)` so trends follow recent behaviour

⏱ ~5 minutes.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')
daily = pd.read_parquet(load.PROCESSED / 'daily_features.parquet')
PF, SAFE = vfm.PHYSICS_FEATURES, vfm.SEPARATOR_SAFE_FEATURES
gp_all = vfm.make_ml_method(PF, vfm.gaussian_process, use_well_id=True, well_encoding='onehot')
gp_safe = vfm.make_ml_method(SAFE, vfm.gaussian_process, use_well_id=True, well_encoding='onehot')
P = vfm.make_physics_method

def weights(oil, water, gas):
    return {'oil_rate_stbd': oil, 'water_rate_stbd': water, 'gas_rate_mscfd': gas}

experiments = {
    'naive: last test value': vfm.time_interpolation,
    'naive: well mean': vfm.well_mean,
    'B physics': vfm.physics,
    'B P_res constant': P(pres_deg=0),
    'B P_res quadratic': P(pres_deg=2),
    'B J fitted': P(j_source='fitted'),
    'B WC last value': P(wc_model='wc_last'),
    'B WC constant': P(wc_model='wc_const'),
    'B slope shrink 0.3': P(pres_shrink=0.3),
    'B slope shrink 0.5': P(pres_shrink=0.5),
    'B recency tau 60 d': P(recency_tau=60),
    'B recency tau 120 d': P(recency_tau=120),
    'A GP (all physics feats)': gp_all,
    'A GP (separator-safe feats)': gp_safe,
    'A Ridge': vfm.make_ml_method(PF, vfm.ridge, True, 'onehot'),
    'A CatBoost': vfm.make_ml_method(PF, vfm.catboost, True, 'onehot'),
    'E2 physics + GP-all water (previous final)': vfm.make_blend_method(vfm.physics, gp_all, weights(0, .5, 0)),
    'F1 rec120 + GP-all oil & water': vfm.make_blend_method(P(recency_tau=120), gp_all, weights(.5, .5, 0)),
    'rec120 + GP-safe oil & water': vfm.make_blend_method(P(recency_tau=120), gp_safe, weights(.5, .5, 0)),
    'F2 rec120 + GP-safe water (FINAL)': vfm.final_method(),
}

forward, loto, preds_forward, preds_loto = [], [], {}, {}
for name, method in experiments.items():
    preds_forward[name], _ = V.run_time_forward(name, tt, method, forward)
    preds_loto[name], _ = V.run_experiment(name, tt, method, loto)

forward = pd.DataFrame(forward)
loto = pd.DataFrame(loto)[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE']].add_prefix('LOTO_')
results = forward.merge(loto.rename(columns={'LOTO_experiment': 'experiment'}), on='experiment')
results.to_csv(load.ROOT / 'outputs' / 'time_forward_results.csv', index=False)
results[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE', 'mean_MAPE_0-30 d',
         'mean_MAPE_30-60 d', 'mean_MAPE_60-92 d', 'LOTO_mean_MAPE']].round(2)

## Why physics oil errs when forecasting: the P_res slope from few tests (psi/month)

In [ ]:
early = vfm.fit_well_trends(tt[tt.time < '2025-06-01'])
full = vfm.fit_well_trends(tt)
pd.DataFrame({'slope_until_June': early.pres_coef.map(lambda c: c[0] * 30.4),
              'slope_all_data': full.pres_coef.map(lambda c: c[0] * 30.4)}).round(1)

## The test-separator effect (a blind spot of validating only on well tests)

During a well test the flow is routed to the **test separator**, which runs at lower pressure (~131 psia).
So flowline (downstream) pressure drops, WHP and Pwf drop a little, and the well produces slightly **more**
than on a normal day. Every validation point is a test, so LOTO and time-forward cannot see this — but the
deliverable predicts **normal days**.

In [ ]:
normal = daily[(daily.on_stream_frac == 1) & daily.dh_pressure_psia.notna()]
cols = ['whp_downstream_psia', 'whp_psia', 'dh_pressure_psia', 'choke_opening_pct']
print('Mean during TESTS minus mean on NORMAL days:')
print((tt.groupby('well_name')[cols].mean() - normal.groupby('well_name')[cols].mean()).round(1))

hf = load.load_high_freq_clean()
t = tt.iloc[0]
w = hf[(hf.well_name == t.well_name) & (hf.timestamp >= t.time - pd.Timedelta(hours=36)) &
       (hf.timestamp <= t.time + pd.Timedelta(hours=36))].set_index('timestamp')
w = w[['whp_downstream_psia', 'whp_psia', 'dh_pressure_psia']].resample('30min').mean()
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, c in zip(axes, w.columns):
    ax.plot(w.index, w[c])
    ax.axvspan(t.time - pd.Timedelta(hours=t.test_hours / 2), t.time + pd.Timedelta(hours=t.test_hours / 2),
               color='orange', alpha=0.2)
    ax.set_ylabel(c, fontsize=8)
axes[0].set_title(f'{t.well_name}: well test (orange) = flow to the test separator')
plt.tight_layout()
plt.show()

In [ ]:
# How does each model behave on NORMAL days? (ratio to physics, which handles the effect through Pwf)
wells = sorted(tt.well_name.unique())
phys = vfm.physics_predict(normal, vfm.fit_well_trends(tt, recency_tau=120))
rows = []
for name, feats in [('GP all physics features', PF), ('GP separator-safe features', SAFE)]:
    for target in ['oil_rate_stbd', 'water_rate_stbd']:
        gp = vfm.gaussian_process().fit(vfm.design_matrix(tt, feats, wells), tt[target])
        ratio = gp.predict(vfm.design_matrix(normal, feats, wells)) / phys[target]
        rows.append({'model': name, 'target': target, 'median GP/physics': np.median(ratio),
                     'p5': np.percentile(ratio, 5), 'p95': np.percentile(ratio, 95)})
pd.DataFrame(rows).round(3)

## Final model F2 vs previous final E2: paired tests

In [ ]:
old, new = 'E2 physics + GP-all water (previous final)', 'F2 rec120 + GP-safe water (FINAL)'
a, b = preds_forward[old], preds_forward[new]
rows = {}
for t in V.TARGETS:
    true = tt.loc[a.test_index, t].values
    d = (abs(b[t].values - true) - abs(a[t].values - true)) / true * 100
    rows[t] = {'mean_diff_pts': d.mean(), 'F2_better_on': int((d < 0).sum()), 'n': len(d),
               'p_value': wilcoxon(d[d != 0]).pvalue if (d != 0).any() else 1.0}
print('FORECASTS (time-forward):')
print(pd.DataFrame(rows).T.round(3))
print('\nLEAVE-ONE-TEST-OUT:')
print(V.paired_test(tt, preds_loto[old], preds_loto[new]).round(3))

In [ ]:
show = ['naive: last test value', 'B physics', 'B recency tau 120 d', 'E2 physics + GP-all water (previous final)',
        'F2 rec120 + GP-safe water (FINAL)']
cols = ['mean_MAPE_0-30 d', 'mean_MAPE_30-60 d', 'mean_MAPE_60-92 d']
ax = results.set_index('experiment').loc[show, cols].T.plot(marker='o', figsize=(8, 4.5))
ax.set_xticks(range(3))
ax.set_xticklabels(['0-30 days', '30-60 days', '60-92 days'])
ax.set_ylabel('mean MAPE % (oil, water, gas)')
ax.set_title('Forecast error vs horizon')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Conclusions**
- Forecasting is harder than interpolating (physics oil 2.4 % → 5.0 %), mainly because the P_res slope from
  5–6 tests is uncertain and some wells' decline accelerates. Physics still beats naive forecasts by far
  (6.2 % vs 9.0 %); P_res decline is essential (constant: 16 %); curved trends / fitted J extrapolate badly.
- **Recency weighting (τ = 120 d)** improves forecasts without hurting interpolation.
- **Test-separator effect:** during tests, flowline pressure is 17–29 psi lower than on normal days → Pwf lower →
  slightly higher rates. A GP given flowline / choke pressures learns this test-only regime and **over-predicts
  normal-day oil by ~10 %** (median GP/physics ratio 1.10 on all normal days). Validating only on tests
  cannot reveal this. A GP restricted to **separator-safe inputs** (downhole P & T, annulus P, lift gas, time, J)
  has no such bias (~1 %).
- **Final model F2:** physics with recency weighting for oil & gas; water = ½ physics + ½ separator-safe GP.
  Forecast error 5.62 % (best), leave-one-test-out 3.87 %, unbiased on normal days.